# Dataset mirror (Kaggle)

Measures whether re-encoding ASL Citizen at short side 256 is worth doing.

Preflight found both architectures spend ~52% of every optimizer step waiting on CPU video decode. This notebook tests the fix on a sample before committing to all 83,399 files.

## Before running

1. **Add Data** → attach the ASL Citizen mirror.
2. **Accelerator** → **None**. This is CPU-only work and must not spend GPU quota.
3. **Internet** → On, so the repository can be cloned.

Then **Run All**. Takes roughly 10-20 minutes.

## The decision this produces

`DECODE SPEEDUP` is the number that matters. Under ~1.6x, the mirror is not worth building and the idea stops here.

## 1. Setup

In [ ]:
import os
import shutil
import subprocess
import sys

assert os.path.exists("/kaggle/input"), (
    "This is the KAGGLE notebook, but this runtime is not Kaggle."
)

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

# Cloned to /tmp: writable, not part of the saved output, discarded with the
# session. Nothing later has to clean it up.
subprocess.run(["rm", "-rf", "/tmp/asl"], check=True)
subprocess.run(["git", "clone", "-q", REPO_URL, "/tmp/asl"], check=True)
PROJECT = "/tmp/asl/ASL_training"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT, "--no-deps"], check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)

ARTIFACTS = "/kaggle/working/artifacts"
OUTPUTS = "/kaggle/working/outputs"
WORK = "/tmp/mirror-calibration"
os.makedirs(OUTPUTS, exist_ok=True)

DATASET_ROOT = None
for attachment in sorted(os.listdir("/kaggle/input")):
    for root, dirs, _files in os.walk(f"/kaggle/input/{attachment}"):
        if "splits" in dirs and "videos" in dirs:
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break
assert DATASET_ROOT, "ASL Citizen not attached. Use Add Data in the sidebar."

assert shutil.which("ffmpeg"), "ffmpeg not found on PATH."


def run(script, **options):
    """Run a project script, streaming its output into this cell.

    Built as an argument list rather than a shell string. IPython's ! escape
    expands $VAR but not {VAR}, and reads $NAME.ext as attribute access, both
    of which have already cost this project a debugging session. subprocess
    has no such rules.
    """
    command = [sys.executable, "-u", f"{PROJECT}/scripts/{script}"]
    for key, value in options.items():
        flag = "--" + key.replace("_", "-")
        # Identity, not equality: 0 == False in Python, so a membership
        # test silently drops --probe-limit 0 and --num-workers 0.
        if value is True:
            command.append(flag)
        elif value is not None and value is not False:
            command += [flag, str(value)]

    print(" ".join(command) + "\n")
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code:
        print(f"\n[exit {code}]")
    return code


print(f"cores      {os.cpu_count()}")
print(f"project    {PROJECT}")
print(f"dataset    {DATASET_ROOT}")

## 2. Manifests

Calibration reads the train manifest to sample clips and to check that frame counts survive re-encoding. `probe_limit=0` skips video probing, so this takes about a minute.

In [ ]:
run(
    "audit_dataset.py",
    dataset_root=DATASET_ROOT,
    output_dir=ARTIFACTS,
    write_manifests=True,
    probe_limit=0,
    expected_classes=2731,
)

## 3. Calibrate

Encodes 300 clips at short side 256, then times decoding through the real loader path — source against mirror, same frame indices, single-threaded so the comparison is fair.

Read three numbers:

1. **DECODE SPEEDUP** — the gate. Under ~1.6x, stop.
2. **projected size** — must fit Kaggle's ~20 GB working directory.
3. **projected encode** — must fit one 12 h CPU session.

`frame mismatches` must be 0. The manifests index against frame counts, so any drift disqualifies the encoding settings.

In [ ]:
CRF = 20

run(
    "calibrate_video_mirror.py",
    dataset_root=DATASET_ROOT,
    artifacts_dir=ARTIFACTS,
    work_dir=WORK,
    samples=300,
    decode_samples=120,
    crf=CRF,
    jobs=os.cpu_count(),
    output=f"{OUTPUTS}/mirror_calibration_crf{CRF}.json",
)

## 4. Optional: compare quality settings

Only worth running if the numbers above are borderline. Lower CRF is higher quality and larger; higher is smaller and lossier. Re-run the cell above with `CRF = 18` or `CRF = 23` and compare the reports.

## 5. Save

**Save Version → Quick Save.** The JSON reports live in `/kaggle/working/outputs/` and are discarded with the session otherwise.